# DBSCAN + EDA + Machine Learning + Prediction

This notebook performs:

- Data loading and inspection
- Exploratory Data Analysis (EDA)
- Missing-value analysis
- Numerical and categorical feature analysis
- Data visualization
- Correlation analysis
- DBSCAN clustering
- DBSCAN parameter analysis using k-distance plot
- Cluster visualization
- Supervised machine-learning model training
- Model evaluation
- Feature importance
- Prediction on new data
- Saving and loading the trained model


In [ ]:
# Install packages if required
# Uncomment the following line if necessary
# !pip install pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
    
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score
)

import joblib

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Configuration

Change `DATA_PATH` to the location of your CSV file and `TARGET_COLUMN` to the column you want to predict.

Example:

```python
DATA_PATH = 'data.csv'
TARGET_COLUMN = 'target'
```


In [ ]:
DATA_PATH = 'data.csv'
TARGET_COLUMN = 'target'

# DBSCAN parameters
DBSCAN_EPS = 0.5
DBSCAN_MIN_SAMPLES = 5

# Random state for reproducibility
RANDOM_STATE = 42

## 2. Load Dataset

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Dataset not found: {DATA_PATH}. Please update DATA_PATH.'
    )

df = pd.read_csv(DATA_PATH)

print('Dataset loaded successfully.')
print('Shape:', df.shape)
display(df.head())

## 3. Basic Dataset Information

In [ ]:
print('Dataset shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())

print('\nData types:')
display(df.dtypes.to_frame('dtype'))

print('\nDataset information:')
df.info()

## 4. Statistical Summary

In [ ]:
display(df.describe(include='all').T)

## 5. Duplicate Records

In [ ]:
duplicate_count = df.duplicated().sum()
print('Number of duplicate rows:', duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print('Duplicates removed.')
    print('New shape:', df.shape)

## 6. Missing-Value Analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_percent = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_table = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage': missing_percent
})

display(missing_table[missing_table['Missing Count'] > 0])

## 7. Identify Numerical and Categorical Columns

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()

print('Numerical columns:')
print(numeric_columns)

print('\nCategorical columns:')
print(categorical_columns)

## 8. Target Distribution

In [ ]:
if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f'{TARGET_COLUMN} does not exist in the dataset. '
        f'Available columns: {df.columns.tolist()}'
    )

print('Target column:', TARGET_COLUMN)
print('Unique values:', df[TARGET_COLUMN].nunique())

display(df[TARGET_COLUMN].value_counts(dropna=False).to_frame('count').head(20))

In [ ]:
plt.figure(figsize=(10, 6))

if df[TARGET_COLUMN].nunique() <= 20:
    sns.countplot(data=df, x=TARGET_COLUMN)
    plt.xticks(rotation=45)
    plt.title('Target Distribution')
else:
    sns.histplot(df[TARGET_COLUMN].dropna(), kde=True)
    plt.title('Target Distribution')

plt.tight_layout()
plt.show()

## 9. Numerical Feature Distributions

In [ ]:
if numeric_columns:
    df[numeric_columns].hist(figsize=(16, 12), bins=30)
    plt.suptitle('Numerical Feature Distributions', fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print('No numerical columns found.')

## 10. Boxplots for Outlier Detection

In [ ]:
if numeric_columns:
    n_cols = 3
    n_rows = int(np.ceil(len(numeric_columns) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = np.array(axes).reshape(-1)

    for i, column in enumerate(numeric_columns):
        sns.boxplot(y=df[column], ax=axes[i])
        axes[i].set_title(f'Boxplot - {column}')

    for i in range(len(numeric_columns), len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

## 11. Correlation Heatmap

In [ ]:
if len(numeric_columns) >= 2:
    plt.figure(figsize=(12, 8))
    correlation = df[numeric_columns].corr()
    sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough numerical columns for correlation analysis.')

## 12. Prepare Data for DBSCAN

DBSCAN works with numerical values, so categorical features are one-hot encoded and numerical features are scaled.

In [ ]:
X = df.drop(columns=[TARGET_COLUMN]).copy()
y = df[TARGET_COLUMN].copy()

feature_numeric = X.select_dtypes(include=np.number).columns.tolist()
feature_categorical = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, feature_numeric),
        ('cat', categorical_transformer, feature_categorical)
    ],
    remainder='drop'
)

X_processed = preprocessor.fit_transform(X)

print('Original feature shape:', X.shape)
print('Processed feature shape:', X_processed.shape)

## 13. DBSCAN k-Distance Plot

The k-distance plot can help choose a reasonable `eps` value. The point around the sharp change in slope is often used as a starting point.

In [ ]:
if len(X_processed) > 5:
    k = min(5, len(X_processed) - 1)

    neighbors = NearestNeighbors(n_neighbors=k)
    neighbors.fit(X_processed)
    distances, indices = neighbors.kneighbors(X_processed)

    k_distances = np.sort(distances[:, -1])

    plt.figure(figsize=(10, 6))
    plt.plot(k_distances)
    plt.title(f'{k}-Distance Plot for DBSCAN')
    plt.xlabel('Data Points sorted by distance')
    plt.ylabel(f'Distance to {k}th nearest neighbor')
    plt.grid(True)
    plt.show()

## 14. Apply DBSCAN

In [ ]:
dbscan = DBSCAN(
    eps=DBSCAN_EPS,
    min_samples=DBSCAN_MIN_SAMPLES
)

cluster_labels = dbscan.fit_predict(X_processed)

df['DBSCAN_Cluster'] = cluster_labels

print('Cluster labels:', np.unique(cluster_labels))
print('\nCluster counts:')
print(df['DBSCAN_Cluster'].value_counts().sort_index())

## 15. DBSCAN Cluster Visualization

For visualization, PCA reduces the processed feature space to two dimensions.

In [ ]:
from sklearn.decomposition import PCA

if X_processed.shape[1] >= 2:
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_processed)

    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=cluster_labels,
        cmap='tab10',
        alpha=0.7
    )

    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.title('DBSCAN Clusters - PCA Visualization')
    plt.colorbar(scatter, label='Cluster')
    plt.show()

    print('Explained variance ratio:', pca.explained_variance_ratio_)
else:
    print('At least two processed features are required for PCA visualization.')

## 16. DBSCAN Silhouette Score

Noise points have label `-1`. The silhouette score is calculated here only when there are at least two non-noise clusters.

In [ ]:
non_noise_mask = cluster_labels != -1
non_noise_labels = cluster_labels[non_noise_mask]

if (
    non_noise_mask.sum() > 1 and
    len(np.unique(non_noise_labels)) >= 2
):
    score = silhouette_score(
        X_processed[non_noise_mask],
        non_noise_labels
    )
    print(f'Silhouette Score excluding noise: {score:.4f}')
else:
    print('Silhouette score cannot be calculated because DBSCAN produced fewer than two non-noise clusters.')

## 17. Prepare Data for Supervised Machine Learning

The DBSCAN cluster is removed from the supervised-learning features because it is an unsupervised result. The target column is used as the prediction target.

In [ ]:
X_ml = df.drop(columns=[TARGET_COLUMN, 'DBSCAN_Cluster']).copy()
y_ml = df[TARGET_COLUMN].copy()

numeric_features = X_ml.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_ml.select_dtypes(exclude=np.number).columns.tolist()

print('ML numerical features:', numeric_features)
print('ML categorical features:', categorical_features)

## 18. Detect Classification vs Regression

The notebook automatically treats a target with object/category/bool type, or a numeric target with relatively few unique values, as classification. Otherwise it uses regression.

In [ ]:
is_classification = (
    y_ml.dtype == 'object' or
    str(y_ml.dtype).startswith('category') or
    y_ml.dtype == 'bool' or
    y_ml.nunique() <= 10
)

if is_classification:
    problem_type = 'classification'
else:
    problem_type = 'regression'

print('Detected problem type:', problem_type)

## 19. Train-Test Split

In [ ]:
if is_classification:
    X_train, X_test, y_train, y_test = train_test_split(
        X_ml,
        y_ml,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y_ml
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X_ml,
        y_ml,
        test_size=0.20,
        random_state=RANDOM_STATE
    )

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

## 20. Build ML Preprocessing Pipeline

In [ ]:
ml_preprocessor = ColumnTransformer(
    transformers=[
        ('num',
         Pipeline([
             ('imputer', SimpleImputer(strategy='median')),
             ('scaler', StandardScaler())
         ]),
         numeric_features),
        ('cat',
         Pipeline([
             ('imputer', SimpleImputer(strategy='most_frequent')),
             ('encoder', OneHotEncoder(handle_unknown='ignore'))
         ]),
         categorical_features)
    ]
)

## 21. Train Random Forest Model

In [ ]:
if is_classification:
    model = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced'
    )
else:
    model = RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

ml_pipeline = Pipeline([
    ('preprocessor', ml_preprocessor),
    ('model', model)
])

ml_pipeline.fit(X_train, y_train)

print('Model training completed.')

## 22. Model Prediction

In [ ]:
y_pred = ml_pipeline.predict(X_test)

prediction_results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred
})

display(prediction_results.head(20))

## 23. Model Evaluation

In [ ]:
if is_classification:
    accuracy = accuracy_score(y_test, y_pred)

    print(f'Accuracy: {accuracy:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

else:
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print(f'MAE:  {mae:.4f}')
    print(f'MSE:  {mse:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'R²:   {r2:.4f}')

    plt.figure(figsize=(8, 6))
    plt.scatter(y_test, y_pred, alpha=0.7)
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Actual vs Predicted')

    min_value = min(np.min(y_test), np.min(y_pred))
    max_value = max(np.max(y_test), np.max(y_pred))
    plt.plot([min_value, max_value], [min_value, max_value], 'r--')

    plt.tight_layout()
    plt.show()

## 24. Feature Importance

In [ ]:
trained_preprocessor = ml_pipeline.named_steps['preprocessor']
trained_model = ml_pipeline.named_steps['model']

feature_names = trained_preprocessor.get_feature_names_out()
importances = trained_model.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

display(importance_df.head(20))

In [ ]:
top_features = importance_df.head(20).sort_values('Importance')

plt.figure(figsize=(10, 8))
plt.barh(top_features['Feature'], top_features['Importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 20 Feature Importances')
plt.tight_layout()
plt.show()

## 25. Predict a New Observation

The following example creates a new observation using the first row of the dataset. Replace it with your own values when making a real prediction.

In [ ]:
# Example new observation
new_data = X_ml.iloc[[0]].copy()

new_prediction = ml_pipeline.predict(new_data)

print('Input:')
display(new_data)

print('Prediction:', new_prediction[0])

## 26. Prediction Probabilities - Classification Only

In [ ]:
if is_classification and hasattr(ml_pipeline.named_steps['model'], 'predict_proba'):
    probabilities = ml_pipeline.predict_proba(new_data)[0]
    classes = ml_pipeline.named_steps['model'].classes_

    probability_df = pd.DataFrame({
        'Class': classes,
        'Probability': probabilities
    }).sort_values('Probability', ascending=False)

    display(probability_df)
else:
    print('Prediction probabilities are available only for classification.')

## 27. Save the Trained Model

In [ ]:
MODEL_PATH = 'trained_random_forest_pipeline.pkl'

joblib.dump(ml_pipeline, MODEL_PATH)

print(f'Model saved to: {MODEL_PATH}')

## 28. Load the Saved Model and Predict

In [ ]:
loaded_model = joblib.load(MODEL_PATH)

loaded_prediction = loaded_model.predict(new_data)

print('Prediction from loaded model:', loaded_prediction[0])

## 29. Save Predictions

In [ ]:
prediction_results.to_csv('predictions.csv', index=False)
print('Predictions saved to predictions.csv')

# Conclusion

This notebook completed the following workflow:

1. Loaded the dataset
2. Performed exploratory data analysis
3. Checked missing values and duplicates
4. Visualized numerical and categorical data
5. Analyzed correlations and outliers
6. Preprocessed the data
7. Applied DBSCAN clustering
8. Visualized DBSCAN clusters using PCA
9. Evaluated the clustering where possible
10. Trained a Random Forest classification or regression model
11. Evaluated the supervised model
12. Examined feature importance
13. Generated predictions
14. Saved and loaded the trained model
